In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

# Force set before any langchain imports
os.environ["LANGCHAIN_API_KEY"] = os.environ.get("LANGCHAIN_API_KEY", "")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "session-16-ragas-eval"


In [2]:
import sys
sys.path.append(".")
from app.rag import _build_rag_graph
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import tiktoken

# Reuse existing Fireworks pipeline
fw_graph = _build_rag_graph("data")

# Build OpenAI pipeline for comparison
def tiktoken_len(text):
    return len(tiktoken.encoding_for_model("gpt-4o").encode(text))

loader = PyMuPDFLoader("data/cat-health-guide.pdf")
chunks = RecursiveCharacterTextSplitter(
    chunk_size=750, chunk_overlap=0, length_function=tiktoken_len
).split_documents(loader.load())

oai_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
oai_vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=oai_embeddings,
    location=":memory:",
    collection_name="oai_eval",
)
oai_retriever = oai_vectorstore.as_retriever()
oai_llm = ChatOpenAI(model="gpt-4o-mini")
prompt = ChatPromptTemplate.from_messages([("human", """
#CONTEXT:
{context}

QUERY:
{query}

Use the provided context to answer the query. If you don't know, say "I don't know".
""")])
oai_chain = prompt | oai_llm | StrOutputParser()

In [3]:
questions = [
    "What vaccinations are recommended for cats?",
    "What should I feed my kitten?",
    "How do I control parasites in cats?",
    "What are signs of illness in cats?",
    "How often should cats visit the vet?",
]

ground_truths = [
    "Cats should receive core vaccines including rabies and FVRCP.",
    "Kittens should be fed high-protein food formulated for their life stage.",
    "Parasite control includes regular flea, tick, and deworming treatments.",
    "Signs of illness include lethargy, loss of appetite, vomiting, and behavioral changes.",
    "Cats should visit the vet at least once a year for wellness exams.",
]

In [4]:
def run_fw_pipeline(questions):
    answers, contexts = [], []
    for q in questions:
        result = fw_graph.invoke({"question": q})
        answers.append(result["response"])
        docs = oai_retriever.invoke(q)  # reuse retriever for context
        contexts.append([d.page_content for d in docs])
    return answers, contexts

def run_oai_pipeline(questions):
    answers, contexts = [], []
    for q in questions:
        docs = oai_retriever.invoke(q)
        answer = oai_chain.invoke({"query": q, "context": docs})
        answers.append(answer)
        contexts.append([d.page_content for d in docs])
    return answers, contexts

print("Running Fireworks pipeline...")
fw_answers, fw_contexts = run_fw_pipeline(questions)

print("Running OpenAI pipeline...")
oai_answers, oai_contexts = run_oai_pipeline(questions)

Running Fireworks pipeline...
Running OpenAI pipeline...


In [5]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
from datasets import Dataset

# Fix wrappers for RAGAS
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", max_tokens=2048))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

def run_ragas(answers, contexts, label):
    dataset = Dataset.from_dict({
        "question": questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truth": ground_truths,
    })
    results = evaluate(
        dataset=dataset,
        metrics=[faithfulness, answer_relevancy, context_recall, context_precision],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
    )
    print(f"\n=== {label} ===")
    print(results)
    return results

fw_results = run_ragas(fw_answers, fw_contexts, "Fireworks OSS")
oai_results = run_ragas(oai_answers, oai_contexts, "OpenAI GPT-4o-mini")

/var/folders/5p/c3nk5nb53p5bkrln79yrc93r0000gn/T/ipykernel_87389/4154476907.py:5: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
/var/folders/5p/c3nk5nb53p5bkrln79yrc93r0000gn/T/ipykernel_87389/4154476907.py:5: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
/var/folders/5p/c3nk5nb53p5bkrln79yrc93r0000gn/T/ipykernel_87389/4154476907.py:5: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.m

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.



=== Fireworks OSS ===
{'faithfulness': 0.7333, 'answer_relevancy': 0.7020, 'context_recall': 1.0000, 'context_precision': 0.9444}


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.



=== OpenAI GPT-4o-mini ===
{'faithfulness': 1.0000, 'answer_relevancy': 0.9088, 'context_recall': 1.0000, 'context_precision': 0.9500}


In [7]:
import pandas as pd

def get_score(results, metric):
    val = results[metric]
    if isinstance(val, list):
        return sum(val) / len(val)
    return float(val)

comparison = pd.DataFrame({
    "Metric": ["Faithfulness", "Answer Relevancy", "Context Recall", "Context Precision"],
    "Fireworks OSS": [
        get_score(fw_results, "faithfulness"),
        get_score(fw_results, "answer_relevancy"),
        get_score(fw_results, "context_recall"),
        get_score(fw_results, "context_precision"),
    ],
    "OpenAI GPT-4o-mini": [
        get_score(oai_results, "faithfulness"),
        get_score(oai_results, "answer_relevancy"),
        get_score(oai_results, "context_recall"),
        get_score(oai_results, "context_precision"),
    ],
})
comparison["Delta"] = comparison["Fireworks OSS"] - comparison["OpenAI GPT-4o-mini"]
print(comparison.to_string(index=False))

           Metric  Fireworks OSS  OpenAI GPT-4o-mini     Delta
     Faithfulness       0.733333            1.000000 -0.266667
 Answer Relevancy       0.701987            0.908781 -0.206795
   Context Recall       1.000000            1.000000  0.000000
Context Precision       0.944444            0.950000 -0.005556


In [13]:
from langsmith import Client
from collections import defaultdict

client = Client()
runs = list(client.list_runs(project_name="session-16-ragas-eval"))

fw_tokens, fw_cost = 0, 0
oai_tokens, oai_cost = 0, 0

for r in runs:
    if r.name in ["RunnableSequence", "LangGraph"]:
        inputs_str = str(r.inputs)
        tokens = r.total_tokens or 0
        cost = r.total_cost or 0
        if "rag_collection" in inputs_str:
            fw_tokens += tokens
            fw_cost += cost
        elif "oai_eval" in inputs_str:
            oai_tokens += tokens
            oai_cost += cost

print(f"Fireworks OSS      — Tokens: {fw_tokens} | Cost: ${fw_cost:.4f}")
print(f"OpenAI GPT-4o-mini — Tokens: {oai_tokens} | Cost: ${oai_cost:.4f}")

Fireworks OSS      — Tokens: 37974 | Cost: $0.0000
OpenAI GPT-4o-mini — Tokens: 38020 | Cost: $0.0065


On retrieval, both pipelines performed equally well. Context Recall was identical at 1.0 and Context Precision was nearly the same (0.944 vs 0.950). However on generation quality, OpenAI GPT-4o-mini significantly outperformed Fireworks OSS — Faithfulness (1.0 vs 0.733) and Answer Relevancy (0.909 vs 0.702) were both notably higher. Combined with lower cost at $0.0065 per 5 queries vs Fireworks fixed $4/hr, OpenAI GPT-4o-mini wins on both quality and cost for low-to-medium volume use cases.